## Tracking

#### Setting Up

In [ ]:
import os
import atracker

# Get the user's home directory
home_dir = os.path.expanduser("~")

# Define the video folder path (inside "test" on the Desktop)
video_folder = os.path.join(home_dir, "Desktop", "tutorial")

# Load ATracker with the video folder
AT = atracker.ATracker(video_folder)

In [ ]:
from atracker import manual_tracker
manual_tracker(media_file="/Volumes/JOL1_4TB/1601 Sociability interaction rules/Pairtrials/3tracked/CM1160116_1130_RP09_S06_G24_TR.mp4",
               data_file="/Volumes/JOL1_4TB/1601 Sociability interaction rules/Pairtrials/3tracked/CM1160116_1130_RP09_S06_G24.csv")

In [ ]:
from atracker import manual_tracker
manual_tracker(media_file="/Users/Jolle/Desktop/tutorial/3tracked/animtest_pair_071024_S01_1150_TR.mp4",
               data_file="/Users/Jolle/Desktop/tutorial/3tracked/animtest_pair_071024_S01_1150.csv")

In [ ]:
from atracker import manual_tracker
manual_tracker(media_file="/Users/Jolle/Dropbox/Science/4 Projects/Up and coming projects - me/PR031 Fission-fusion experiment/barcoding/grouptest_barcodes_250425_S01_1753_fr1.jpg",
               data_file="/Users/Jolle/Dropbox/Science/4 Projects/Up and coming projects - me/PR031 Fission-fusion experiment/barcoding/grouptest_barcodes_250425_S01_1753_fr1.csv")

#### Checking the config file 

After you have finished setting all parameters, double check the overview file and configuration file to see if indeed everything is as desired. 

In particular, there are many tracking settings that you may want to adjust, including if a tracking video should be created (`create_vid`), if data should be created (`create_dat`), if videos should be overwritten if tracked again (`overwrite`), among others, as well as many display settings, including if tracking should be shown in realtime (`show_tracking`), the size of the display window (`vid_displaysize`) and drawing parameters.

An important tracking parameter to consider is `simple`. When set to True, tracking will only try to extract the centroid coordinates of the contours. When False, it will much more accurately try to track the shape, including its orientation, and create a skeleton to create key points of the shape, including head, centroid, and tail. This is more computationally intensive and in most cases is not needed.

In [ ]:
AT.set_config(show_tracking=True, vid_displaysize=1, frame_disstep=10, userwait=False, simple=True, 
              create_vid=True, create_dat=True, overwrite=True)

#### Excluding files

There are various ways to select the videos to track. A simple one is to set the `folder` parameter to the `todo` folder and then only videos in that folder will be tracked. A more robust method to exclude videos where e.g. something was wrong during the trial or recording is to use the `exclude` column in the overview file. Any rows that have exclude = 1 will be excluded from tracking.

#### Drymode

Before starting the actual tracking, it may be a good idea to run a so-called "dryrun" using the `drymode()` function. This function will randomly take sections of a random selection of videos and track them so it is quick and easy to see if any potential tracking issues will come up as they should also become apparent in these short fragments. 

You can set the `rand_filenr`, which is the number of random video files that should be selected, `rand_seqnr`, the number of video fragments from each video file, `rand_seqlen`, the length of each video fragment, `suffix`, a potential suffix for the filenames to indicate they are created using drymode, and `rerun`, which is to use the same randomly selected indices of videos files and sequences, which can be helpful to run drymode again on the same videos after some tweaks to thresholding.

In [ ]:
AT.drymode(rand_filenr=7, rand_seqnr=1, rand_seqlen=50, suffix="dry", rerun=True)

#### Tracking! 

Now, to start tracking we can simply use the `track` function. Here again the `inds`, `query` and `cats` parameters can be used. 

It is also possible to override the start and stop frames, set custom threshtypes, and add a suffix to the video. These latter parameters might be helpful to track part of some videos to test some idea for example without having to touch the overview file.

In [ ]:
AT.track(inds=None, names=None, query=None, cats=None, pools=1, folder="todo", threshtype=None, suffix="")

You can provide specific indices of files or a list of filenames to only track a subset of videos, e.g.

In [ ]:
AT.track(inds=[7], folder="originals")

or

In [ ]:
AT.track(names=["FF_bol_T2_200818_jolpi105_113141_S08", "FF_bol_T2_200818_jolpi105_130410_S10"])

In [ ]:
AT.set_config(show_tracking=False, simple=False, create_vid=True, create_dat=True, overwrite=True)
AT.track(names="animtest_pair_071024_S01_1150", folder="originals", stop=300)

#### Running Parallel Tracking with the pools Parameter

The pools parameter in `AT.track()` enables running multiple tracking jobs in parallel, making use of all available CPU cores. For example, setting `pools=8` will attempt to run up to 8 tracking processes at once on an 8-core machine, significantly speeding up batch processing.

> **Note**:
    Multiprocessing cannot be used interactively in Jupyter Notebooks. To use `pools > 1`, you must run your tracking code from a standard Python script in the terminal.

**How to run parallel tracking**
1. **Create a Python file (e.g., poolatracker.py) in your tracking directory.**
   
2. **Use the following code template:**

In [ ]:
import atracker
import os

def main():
    # Set the path to your tracking folder
    home_dir = os.path.expanduser("~")
    video_folder = os.path.join(home_dir, "Desktop", "tutorial")

    # Initialize ATracker
    AT = atracker.ATracker(video_folder)
    AT.set_config(overwrite=True)
    AT.track(folder="originals", pools=4)  # Adjust pools as needed

if __name__ == "__main__":
    # This guard is required for multiprocessing to work safely on macOS/Windows
    main()

**3. Run from the terminal:**

- Activate your Python virtual environment.
- Navigate to your tracking folder (cd `TRACKINGDIR`).
- Run the script: `python poolatracker.py`

Why use if __name__ == "__main__"?

This guard is required on macOS and Windows for multiprocessing.
Without it, each process would re-run your entire script, leading to errors or infinite loops.
Always place your main code block under this guard when using pooled tracking.3

#### Manual tracking

To be added..

#### Processing

In [ ]:
AT.process(
    pools=1,
    names=["animtest_solo_071024_S01_1413"],
    overwrite=True,
    fulldata=True,
    convert=True,
    smoothwin=10,
    mask_margin=20,
    max_traj_gap=50,
    min_traj_len=10,
    roi_edge_margin=10,
    interp_gap_com=500,
    interp_gap_orient=100,
    orient_min_speed=1,
    interpolate=True,
    compute_movement=True,
    compute_distances=True,
)